# text_analysis.ipynb

This notebook does basic topic modeling of the data in r/WomensHealth

In [1]:
# importing os module for environment variables
import os
# importing necessary functions from dotenv library
from dotenv import load_dotenv 
# loading variables from .env file
load_dotenv()

True

In [2]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DBUSER')}:{os.getenv('DBPASSWORD')}"
    f"@{os.getenv('DBHOST')}:{os.getenv('DBPORT')}/{os.getenv('DBNAME')}"
)

# Whole table
df = pd.read_sql("SELECT * FROM submissions", engine)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123997 entries, 0 to 123996
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            123997 non-null  object 
 1   title         123997 non-null  object 
 2   selftext      123997 non-null  object 
 3   upvote_ratio  106024 non-null  float64
 4   ups           65710 non-null   float64
 5   num_comments  123997 non-null  int64  
 6   subreddit     123997 non-null  object 
dtypes: float64(2), int64(1), object(4)
memory usage: 6.6+ MB


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123997 entries, 0 to 123996
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            123997 non-null  object 
 1   title         123997 non-null  object 
 2   selftext      123997 non-null  object 
 3   upvote_ratio  106024 non-null  float64
 4   ups           65710 non-null   float64
 5   num_comments  123997 non-null  int64  
 6   subreddit     123997 non-null  object 
dtypes: float64(2), int64(1), object(4)
memory usage: 6.6+ MB


## Modeling

In [ ]:
# pip install bertopic pandas sentence-transformers umap-learn hdbscan --break-system-packages

In [4]:
# prep the text

import re

df = df.copy()
df['selftext'] = df['selftext'].fillna('')

# Drop posts where selftext is just removed/deleted boilerplate
junk = {'[removed]', '[deleted]', ''}
df['selftext_clean'] = df['selftext'].apply(lambda x: '' if x.strip() in junk else x)

# Combine title + body — title carries a lot of signal in Reddit posts
df['text'] = (df['title'].str.strip() + '. ' + df['selftext_clean'].str.strip()).str.strip()

# Basic cleanup: strip URLs, excess whitespace
df['text'] = df['text'].apply(lambda x: re.sub(r'http\S+|www\.\S+', '', x))
df['text'] = df['text'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())

# Drop rows with too little text to be meaningful
df = df[df['text'].str.len() > 15].reset_index(drop=True)
print(len(df), "posts after cleaning")

120867 posts after cleaning


In [5]:
# run BERTopic

from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# Remove boilerplate / stopwords-ish tokens specific to Reddit
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=5)

topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    min_topic_size=50,      # tune based on dataset size — larger = fewer, broader topics
    nr_topics="auto",       # let it merge similar topics automatically
    calculate_probabilities=False,  # faster; turn on if you need per-doc confidence
    verbose=True
)

topics, _ = topic_model.fit_transform(df['text'].tolist())
df['topic'] = topics

2026-06-22 18:08:31,447 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3778 [00:00<?, ?it/s]

2026-06-22 18:54:39,646 - BERTopic - Embedding - Completed ✓
2026-06-22 18:54:39,653 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-22 18:56:33,668 - BERTopic - Dimensionality - Completed ✓
2026-06-22 18:56:33,674 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-22 18:56:55,529 - BERTopic - Cluster - Completed ✓
2026-06-22 18:56:55,532 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-06-22 18:57:25,235 - BERTopic - Representation - Completed ✓
2026-06-22 18:57:25,258 - BERTopic - Topic reduction - Reducing number of topics
2026-06-22 18:57:25,422 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-22 18:57:52,731 - BERTopic - Representation - Completed ✓
2026-06-22 18:57:52,770 - BERTopic - Topic reduction - Reduced number of topics from 226 to 14


In [6]:
# inspect results
# Overview: topic id, count, top keywords
topic_info = topic_model.get_topic_info()
print(topic_info.head(20))

# Top words for a specific topic
topic_model.get_topic(0)

# Most frequent topics overall (-1 = outliers/no clear topic, expect 10-30% here)
df['topic'].value_counts().head(20)

    Topic  Count                                       Name  \
0      -1  45272                        -1_im_help_need_use   
1       0  72631                         0_im_help_need_use   
2       1    861                            1_10_30_need_im   
3       2    707   2_user deleted_deleted user_deleted_user   
4       3    353   3_user deleted_deleted user_deleted_user   
5       4    264   4_user deleted_deleted user_deleted_user   
6       5    168   5_user deleted_deleted user_deleted_user   
7       6    144                     6_help_need_hello_link   
8       7    116   7_user deleted_deleted user_deleted_user   
9       8    107   8_user deleted_deleted user_deleted_user   
10      9     74                         9_link_30_10_hello   
11     10     60                                     10____   
12     11     58                        11_link_10_30_hello   
13     12     52  12_user deleted_deleted user_deleted_user   

                                       Representation 

topic
 0     72631
-1     45272
 1       861
 2       707
 3       353
 4       264
 5       168
 6       144
 7       116
 8       107
 9        74
 10       60
 11       58
 12       52
Name: count, dtype: int64

In [ ]:
# visualize
topic_model.visualize_barchart(top_n_topics=20)   # keyword bars per topic
topic_model.visualize_topics()                     # 2D topic similarity map
topic_model.visualize_documents(df['text'].tolist())  # doc-level scatter, can be slow at 124k

In [7]:
type(topic_model)

bertopic._bertopic.BERTopic

In [8]:
type(topic_info)

pandas.core.frame.DataFrame

In [9]:
topic_info.to_csv("../data/processed/topic_info_0.csv")